# Karman Nowcasting Model — Inference Demo

This notebook walks through running inference with a pre-trained **Karman nowcasting model** that predicts thermospheric neutral density (kg/m³) from space weather and orbital parameters.

> **Self-contained**: This notebook runs entirely in Google Colab with no local dependencies — the model class, sample data, and weights are all included or downloaded at runtime.

## How it works

The model is a simple feedforward neural network (MLP) that learns a **residual correction** on top of an exponential atmosphere baseline:

$$\rho_{\text{pred}} = \text{unscale}\!\Big(\text{scale}(\rho_{\text{expo}}) + \tanh\!\big(\text{NN}(\mathbf{x})\big)\Big)$$

where:
- $\rho_{\text{expo}}$ is the density from a simple exponential atmosphere model (a rough first guess),
- $\mathbf{x}$ is a vector of 18 space-weather and orbital features (already normalized),
- The NN outputs a correction in normalized log-space, squeezed through $\tanh$ to keep it bounded,
- `scale` / `unscale` convert between physical density and the normalized $[-1, 1]$ log-space used during training.

# Karman Nowcasting Model — Inference Demo

This notebook walks through running inference with a pre-trained **Karman nowcasting model** that predicts thermospheric neutral density (kg/m³) from space weather and orbital parameters.

> **Self-contained**: This notebook runs entirely in Google Colab with no local dependencies — the model class, sample data, and weights are all included or downloaded at runtime.

## How it works

The model is a simple feedforward neural network (MLP) that learns a **residual correction** on top of an exponential atmosphere baseline:

$$\rho_{\text{pred}} = \text{unscale}\!\Big(\text{scale}(\rho_{\text{expo}}) + \tanh\!\big(\text{NN}(\mathbf{x})\big)\Big)$$

where:
- $\rho_{\text{expo}}$ is the density from a simple exponential atmosphere model (a rough first guess),
- $\mathbf{x}$ is a vector of 18 space-weather and orbital features (already normalized),
- The NN outputs a correction in normalized log-space, squeezed through $\tanh$ to keep it bounded,
- `scale` / `unscale` convert between physical density and the normalized $[-1, 1]$ log-space used during training.

# Karman Nowcasting Model — Inference Demo

This notebook walks through running inference with a pre-trained **Karman nowcasting model** that predicts thermospheric neutral density (kg/m³) from space weather and orbital parameters.

> **Self-contained**: This notebook runs entirely in Google Colab with no local dependencies — the model class, sample data, and weights are all included or downloaded at runtime.

## How it works

The model is a simple feedforward neural network (MLP) that learns a **residual correction** on top of an exponential atmosphere baseline:

$$\rho_{\text{pred}} = \text{unscale}\!\Big(\text{scale}(\rho_{\text{expo}}) + \tanh\!\big(\text{NN}(\mathbf{x})\big)\Big)$$

where:
- $\rho_{\text{expo}}$ is the density from a simple exponential atmosphere model (a rough first guess),
- $\mathbf{x}$ is a vector of 18 space-weather and orbital features (already normalized),
- The NN outputs a correction in normalized log-space, squeezed through $\tanh$ to keep it bounded,
- `scale` / `unscale` convert between physical density and the normalized $[-1, 1]$ log-space used during training.

## 1. Install dependencies

In [ ]:
!pip install -q torch matplotlib

## 2. Imports and model definition

The `SimpleNetwork` is a generic fully-connected MLP (from the [Karman](https://github.com/FrontierDevelopmentLab/2024-HL-Thermo-CL) codebase). We define it inline here so the notebook is self-contained.

In [ ]:
import json, io
import torch
from torch import nn
from urllib.request import urlopen
import matplotlib.pyplot as plt

torch.set_default_dtype(torch.float32)


class SimpleNetwork(nn.Module):
    """Generic fully connected MLP with adjustable depth."""

    def __init__(
        self,
        input_dim,
        hidden_layer_dims=[256, 256, 256],
        output_dim=1,
        act=nn.LeakyReLU(negative_slope=0.01),
        dropout=0.0,
    ):
        super(SimpleNetwork, self).__init__()
        dims = [input_dim] + hidden_layer_dims
        self.dropout = nn.Dropout(p=dropout)
        self.fcs = nn.ModuleList(
            [nn.Linear(dims[i], dims[i + 1]) for i in range(len(dims) - 1)]
        )
        self.act = act
        self.acts = nn.ModuleList([self.act for _ in range(len(dims) - 1)])
        self.fc_out = nn.Linear(dims[-1], output_dim)

    def forward(self, x):
        for fc, act in zip(self.fcs, self.acts):
            x = act(fc(self.dropout(x)))
        return self.fc_out(x)

## 3. Load sample data

The demo data contains:
- **`normalization_dict`** — min/max of log₁₀(density) used to map densities to [−1, 1] during training.
- **`feature_names`** — the 18 input features (altitude, latitude, solar indices, geomagnetic indices, and cyclical encodings of longitude / day-of-year / time-of-day).
- **`samples`** — a list of pre-processed data points, each with:
  - `instantaneous_features`: the 18 normalized input values,
  - `exponential_atmosphere`: the baseline exponential-model density (kg/m³),
  - `ground_truth`: the observed density (kg/m³).

In [ ]:
data = {
  "normalization_dict": {
    "log_exp_residual": {
      "max": -9.80734920501709,
      "min": -13.99995231628418
    }
  },
  "feature_names": [
    "tudelft_thermo__altitude__[m]",
    "tudelft_thermo__latitude__[deg]",
    "celestrack__ap_average__",
    "space_environment_technologies__f107_obs__",
    "space_environment_technologies__f107_average__",
    "space_environment_technologies__s107_obs__",
    "space_environment_technologies__s107_average__",
    "space_environment_technologies__m107_obs__",
    "space_environment_technologies__m107_average__",
    "space_environment_technologies__y107_obs__",
    "space_environment_technologies__y107_average__",
    "JB08__d_st_dt__[K]",
    "tudelft_thermo__longitude__[deg]_sin",
    "tudelft_thermo__longitude__[deg]_cos",
    "all__day_of_year__[d]_sin",
    "all__day_of_year__[d]_cos",
    "all__seconds_in_day__[s]_sin",
    "all__seconds_in_day__[s]_cos"
  ],
  "samples": [
    {
      "instantaneous_features": [0.4965, 0.3186, -0.8007, -0.1492, 0.4357, 0.5696, 0.7742, 0.1767, 0.5971, 0.7113, 0.7433, -0.4697, 0.0202, 0.9998, -0.4559, -0.8900, 0.2705, 0.9627],
      "exponential_atmosphere": 1.2454e-12,
      "ground_truth": 1.2774e-12
    },
    {
      "instantaneous_features": [0.4278, 0.0661, -0.8007, -0.1492, 0.4357, 0.5696, 0.7742, 0.1767, 0.5971, 0.7113, 0.7433, -0.4697, 0.0153, 0.9999, -0.4559, -0.8900, 0.2950, 0.9555],
      "exponential_atmosphere": 1.4929e-12,
      "ground_truth": 1.6861e-12
    },
    {
      "instantaneous_features": [0.4187, 0.0227, -0.8007, -0.1492, 0.4357, 0.5696, 0.7742, 0.1767, 0.5971, 0.7113, 0.7433, -0.4697, 0.0142, 0.9999, -0.4559, -0.8900, 0.2991, 0.9542],
      "exponential_atmosphere": 1.5289e-12,
      "ground_truth": 1.7021e-12
    },
    {
      "instantaneous_features": [0.3963, -0.6501, -0.8007, -0.1492, 0.4357, 0.5696, 0.7742, 0.1767, 0.5971, 0.7113, 0.7433, -0.4697, 0.1992, -0.9800, -0.4559, -0.8900, 0.4297, 0.9030],
      "exponential_atmosphere": 1.6234e-12,
      "ground_truth": 2.7561e-12
    },
    {
      "instantaneous_features": [0.3904, -0.5422, -0.8007, -0.1492, 0.4357, 0.5696, 0.7742, 0.1767, 0.5971, 0.7113, 0.7433, -0.4697, 0.1880, -0.9822, -0.4559, -0.8900, 0.4395, 0.8982],
      "exponential_atmosphere": 1.6501e-12,
      "ground_truth": 2.8184e-12
    }
  ]
}

log_min = data['normalization_dict']['log_exp_residual']['min']
log_max = data['normalization_dict']['log_exp_residual']['max']
feature_names = data['feature_names']
samples = data['samples']

print(f"Normalization range: log10(density) ∈ [{log_min:.4f}, {log_max:.4f}]")
print(f"Number of features: {len(feature_names)}")
print(f"Number of samples:  {len(samples)}")
print(f"\nFeatures:")
for i, name in enumerate(feature_names):
    print(f"  [{i:2d}] {name}")

## 4. Define the scaling functions

Because density spans many orders of magnitude (~10⁻¹⁴ to 10⁻¹⁰ kg/m³), the model works in **normalized log-space**. Training targets are mapped to [−1, 1] via:

$$\text{scaled} = 2 \cdot \frac{\log_{10}(\rho) - \log_{\min}}{\log_{\max} - \log_{\min}} - 1$$

and the inverse:

$$\rho = 10^{\;\frac{(\log_{\max} - \log_{\min})(\text{scaled} + 1)}{2} \;+\; \log_{\min}}$$

In [ ]:
def scale_density(density, log_min, log_max):
    """Map a physical density (kg/m³) into normalized log-space [-1, 1]."""
    tmp = torch.log10(density)
    return 2.0 * (tmp - log_min) / (log_max - log_min) - 1.0


def unscale_density(scaled, log_min, log_max):
    """Map a normalized log-space value back to physical density (kg/m³)."""
    tmp = (log_max - log_min) * (scaled + 1) / 2 + log_min
    return torch.pow(10, tmp)


# Quick sanity check: round-trip a known density value
test_rho = torch.tensor(1.5e-12)
reconstructed = unscale_density(scale_density(test_rho, log_min, log_max), log_min, log_max)
print(f"Original:      {test_rho.item():.6e}")
print(f"Round-tripped:  {reconstructed.item():.6e}")
print(f"Match: {torch.allclose(test_rho, reconstructed)}")

## 5. Load the pre-trained model from S3

The model weights are hosted on a **public S3 bucket** — no AWS credentials required. We download them into memory at runtime (~140 KB).

The model is a `SimpleNetwork` — a fully-connected MLP with 3 hidden layers of 128 units each and LeakyReLU activations:

```
Input (18) → [Linear → LeakyReLU] × 3 → Linear → Output (1)
```

In [ ]:
MODEL_URL = (
    "https://nasa-radiant-data.s3.amazonaws.com/"
    "helioai-datasets/hl-therm/models/"
    "karman_nowcast_model_log_exp_residual_valid_mape_15.14_params_35585.torch"
)
HIDDEN_LAYER_DIM = 128
HIDDEN_LAYERS = 3

# Download weights into memory (small model — ~140 KB)
print("Downloading model weights from S3 …")
with urlopen(MODEL_URL) as resp:
    model_bytes = resp.read()
print(f"Downloaded {len(model_bytes) / 1024:.1f} KB")

n_features = len(feature_names)
model = SimpleNetwork(
    input_dim=n_features,
    act=torch.nn.LeakyReLU(negative_slope=0.01),
    hidden_layer_dims=[HIDDEN_LAYER_DIM] * HIDDEN_LAYERS,
    output_dim=1,
)
model.load_state_dict(torch.load(io.BytesIO(model_bytes), map_location='cpu', weights_only=True))
model.eval()

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model loaded: {n_params:,} trainable parameters")
print(f"\nArchitecture:\n{model}")

## 6. Run inference on all samples

For each sample, the prediction pipeline is:

1. **Forward pass** — feed the 18 features through the MLP to get a raw correction value.
2. **Bound with tanh** — apply `tanh` so the correction stays in (−1, 1), preventing wild extrapolation.
3. **Add to baseline** — scale the exponential-atmosphere density to log-space and add the NN correction.
4. **Unscale** — convert back from normalized log-space to physical density (kg/m³).

We measure accuracy using **Mean Absolute Percentage Error (MAPE)**.

In [ ]:
predictions = []
ground_truths = []
expo_baselines = []
mapes = []

with torch.no_grad():
    for s in samples:
        features = torch.tensor(s['instantaneous_features']).unsqueeze(0)  # (1, 18)
        expo_atm = torch.tensor(s['exponential_atmosphere'])
        gt = s['ground_truth']

        # Step 1 & 2: NN forward pass + tanh bounding
        raw_correction = model(features).squeeze()
        bounded_correction = torch.tanh(raw_correction)

        # Step 3: Add correction to scaled baseline
        scaled_baseline = scale_density(expo_atm, log_min, log_max)
        scaled_prediction = scaled_baseline + bounded_correction

        # Step 4: Convert back to physical density
        density_pred = unscale_density(scaled_prediction, log_min, log_max).item()

        predictions.append(density_pred)
        ground_truths.append(gt)
        expo_baselines.append(s['exponential_atmosphere'])
        mapes.append(abs(density_pred - gt) / gt * 100)

# Print results table
print(f"{'Sample':>6}  {'Predicted [kg/m³]':>20}  {'Ground Truth [kg/m³]':>22}  {'MAPE [%]':>10}")
print("-" * 65)
for i, (pred, gt, mape) in enumerate(zip(predictions, ground_truths, mapes)):
    print(f"{i:>6}  {pred:>20.6e}  {gt:>22.6e}  {mape:>10.2f}")
print("-" * 65)
print(f"Mean Absolute Percentage Error: {sum(mapes) / len(mapes):.2f}%")

## 7. Visualize results

### Predicted vs. ground truth

A perfect model would place all points on the diagonal line. We also plot the exponential-atmosphere baseline to show how much the NN correction improves things.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

ax.scatter(ground_truths, expo_baselines, alpha=0.6, label='Exponential atmosphere (baseline)', marker='x', color='gray')
ax.scatter(ground_truths, predictions, alpha=0.7, label='Karman NN prediction', marker='o', color='tab:blue', edgecolors='k', linewidths=0.3)

# Perfect-prediction diagonal
lo = min(min(ground_truths), min(predictions)) * 0.8
hi = max(max(ground_truths), max(predictions)) * 1.2
ax.plot([lo, hi], [lo, hi], 'k--', lw=1, label='Perfect prediction')

ax.set_xlabel('Ground truth density [kg/m³]')
ax.set_ylabel('Predicted density [kg/m³]')
ax.set_title('Predicted vs. Ground Truth Density')
ax.legend()
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

### Per-sample MAPE

This bar chart shows the percentage error for each sample. Lower is better.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(range(len(mapes)), mapes, color='tab:orange', edgecolor='k', linewidth=0.3)
ax.axhline(sum(mapes) / len(mapes), color='tab:red', ls='--', lw=1.5, label=f'Mean MAPE = {sum(mapes)/len(mapes):.1f}%')
ax.set_xlabel('Sample index')
ax.set_ylabel('MAPE [%]')
ax.set_title('Per-Sample Absolute Percentage Error')
ax.legend()
plt.tight_layout()
plt.show()